In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

In [2]:
data = pd.read_csv("data_for_modeling.csv")
data.head()

,points_share,VORP,PER,FG,2P,DWS,USG%,WS/48,WS,PTS,FTA
0,0.000,1.7,15.9,5.8,3.0,1.8,21.7,0.150,6.6,17.1,3.1
1,0.000,1.3,20.9,3.0,2.9,1.7,21.7,0.168,3.5,6.8,1.1
2,0.000,0.0,2.9,0.3,0.3,0.0,20.0,-0.083,0.0,1.3,0.7
3,0.056,5.5,24.4,6.7,5.2,2.9,24.2,0.211,10.7,18.5,4.2
4,0.000,4.2,20.6,7.4,4.5,2.0,27.4,0.150,8.6,22.1,5.3


In [3]:
data.describe()

,points_share,VORP,PER,FG,2P,DWS,USG%,WS/48,WS,PTS,FTA
count,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000
mean,0.017643,0.581933,12.825817,3.077672,2.387202,1.195597,18.570237,0.074675,2.482539,8.270859,1.889832
std,0.108413,1.289836,6.215327,2.210685,1.895446,1.136992,5.388191,0.098156,2.832190,6.093909,1.737911
min,0.000000,-2.000000,-54.400000,0.000000,0.000000,-0.600000,0.000000,-1.312000,-2.100000,0.000000,0.000000
25%,0.000000,-0.100000,10.000000,1.400000,1.000000,0.300000,15.000000,0.040000,0.300000,3.700000,0.700000
50%,0.000000,0.100000,12.800000,2.500000,1.900000,0.900000,18.100000,0.082000,1.600000,6.700000,1.400000
75%,0.000000,0.900000,15.900000,4.300000,3.300000,1.800000,21.700000,0.120000,3.800000,11.500000,2.500000
max,1.000000,11.800000,133.800000,12.200000,11.600000,9.100000,72.500000,2.712000,20.300000,36.100000,13.100000


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12697 entries, 0 to 12696
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   points_share  12697 non-null  float64
 1   VORP          12697 non-null  float64
 2   PER           12697 non-null  float64
 3   FG            12697 non-null  float64
 4   2P            12697 non-null  float64
 5   DWS           12697 non-null  float64
 6   USG%          12697 non-null  float64
 7   WS/48         12697 non-null  float64
 8   WS            12697 non-null  float64
 9   PTS           12697 non-null  float64
 10  FTA           12697 non-null  float64
dtypes: float64(11)
memory usage: 1.1 MB


In [11]:
# Assign target variable
y = data["points_share"]

In [21]:
# Assign features
X = data[['VORP', 'PER', 'FG', '2P', 'DWS', 'USG%', 'WS/48', 'WS', 'PTS', 'FTA']]

In [13]:
# Import StandardScaler
from sklearn.preprocessing import StandardScaler

In [14]:
# For newer versions of Python/Jupyter, change default so that method config attributes always show
from sklearn import set_config
set_config(print_changed_only=False, display="diagram")

In [23]:
# Scale data
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_scaled.head()

,VORP,PER,FG,2P,DWS,USG%,WS/48,WS,PTS,FTA
0,0.866863,0.494633,1.231489,0.323313,0.531601,0.580879,0.767434,1.453865,1.448904,0.696362
1,0.556734,1.299127,-0.035136,0.270553,0.443646,0.580879,0.950823,0.359263,-0.241375,-0.454490
2,-0.451186,-1.597053,-1.256525,-1.101210,-1.051586,0.265362,-1.606439,-0.876578,-1.143951,-0.684661
3,3.813089,1.862273,1.638619,1.484035,1.499104,1.044875,1.388919,2.901565,1.678651,1.329331
4,2.805169,1.250858,1.955275,1.114715,0.707511,1.638790,0.767434,2.160060,2.269428,1.962300


In [25]:
# Import VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [27]:
# Remove VIF Iteratively

def calculate_vif(df, thresh=10.0):
    variables = df.columns.tolist()
    dropped = True
    while dropped:
        dropped = False
        vif_data = pd.DataFrame()
        vif_data["Feature"] = variables
        vif_data["VIF"] = [variance_inflation_factor(df[variables].values, i) for i in range(len(variables))]

        max_vif = vif_data["VIF"].max()
        if max_vif > thresh:
            drop_feature = vif_data.sort_values("VIF", ascending=False)["Feature"].iloc[0]
            print(f"Dropping '{drop_feature}' with VIF: {max_vif:.2f}")
            variables.remove(drop_feature)
            dropped = True

# Usage
cleaned_df = calculate_vif(X_scaled, thresh=10)

Dropping 'FG' with VIF: 887.84
Dropping 'PER' with VIF: 18.45
Dropping 'WS' with VIF: 12.32


In [31]:
# Show final columns for modeling along with VIF
X_scaled_final = X_scaled[["USG%", "DWS", "VORP", "WS/48", "FTA", "2P", "PTS"]]

vif_final = pd.DataFrame()
vif_final["Features"] = X_scaled_final.columns
vif_final["VIF"] = [variance_inflation_factor(X_scaled_final.values, i) for i in range(X_scaled_final.shape[1])]

vif_final.sort_values(by="VIF", ascending=True)

,Features,VIF
3,WS/48,1.385923
0,USG%,1.979146
1,DWS,2.578201
2,VORP,3.122776
4,FTA,5.067715
5,2P,7.020389
6,PTS,8.144588


In [40]:
# Import Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_regression
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

In [46]:
# Run regression 10 times with random seed value each time
results = []
p_values_matrix = pd.DataFrame(index=X_scaled_final.columns)
weights_matrix = pd.DataFrame(index=X_scaled_final.columns)

num_runs = 10

for i in range(num_runs):
    seed = np.random.randint(1, 100)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled_final, y, test_size=0.2, random_state=seed)

    # Linear regression
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict and log results
    y_pred = model.predict(X_test)
    results.append({
        "Run_Number" : i + 1,
        "Random_State" : seed,
        "R_Squared" : round(r2_score(y_test, y_pred), 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 4),
        "RMSE": round(root_mean_squared_error(y_test, y_pred), 4)
    })
    p_value = f_regression(X_train, y_train)[1]
    weights = model.coef_

    # Store results
    p_values_matrix[f'run_{i+1}'] = pd.Series(p_value, index=X_train.columns)
    weights_matrix[f'run_{i+1}'] = pd.Series(weights, index=X_train.columns)

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df

,Run_Number,Random_State,R_Squared,MAE,RMSE
0,1,51,0.4193,0.0412,0.0826
1,2,23,0.4816,0.0382,0.0752
2,3,35,0.4146,0.0411,0.0809
3,4,5,0.4487,0.0412,0.0845
4,5,57,0.4103,0.0409,0.0835
5,6,80,0.4370,0.0396,0.0794
6,7,86,0.4005,0.0425,0.0832
7,8,87,0.4304,0.0411,0.0810
8,9,7,0.4395,0.0402,0.0806
9,10,25,0.4114,0.0402,0.0767


In [48]:
# P-values and weights
combined_summary = pd.DataFrame({
    'Avg_Weight': weights_matrix.mean(axis=1).round(4),
    'Avg_p_value': p_values_matrix.mean(axis=1).round(4)
}).sort_values(by='Avg_Weight', ascending=False)

combined_summary

,Avg_Weight,Avg_p_value
VORP,0.0798,0.0
FTA,0.0310,0.0
USG%,0.0076,0.0
2P,0.0060,0.0
WS/48,-0.0047,0.0
DWS,-0.0131,0.0
PTS,-0.0389,0.0
